In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
results_df = pd.read_csv("results.csv")
rankings_df = pd.read_csv("fifa_ranking-2024-06-20.csv")

EDA & preprocessing

In [3]:
results_df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [4]:
rankings_df.head()

,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,140.0,Brunei Darussalam,BRU,2.0,0.0,140,AFC,1992-12-31
1,33.0,Portugal,POR,38.0,0.0,33,UEFA,1992-12-31
2,32.0,Zambia,ZAM,38.0,0.0,32,CAF,1992-12-31
3,31.0,Greece,GRE,38.0,0.0,31,UEFA,1992-12-31
4,30.0,Algeria,ALG,39.0,0.0,30,CAF,1992-12-31


In [5]:
results_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49477 entries, 0 to 49476
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49477 non-null  str    
 1   home_team   49477 non-null  str    
 2   away_team   49477 non-null  str    
 3   home_score  49433 non-null  float64
 4   away_score  49433 non-null  float64
 5   tournament  49477 non-null  str    
 6   city        49477 non-null  str    
 7   country     49477 non-null  str    
 8   neutral     49477 non-null  bool   
dtypes: bool(1), float64(2), str(6)
memory usage: 3.1 MB


In [6]:
rankings_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 67472 entries, 0 to 67471
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   rank             67463 non-null  float64
 1   country_full     67472 non-null  str    
 2   country_abrv     67472 non-null  str    
 3   total_points     67472 non-null  float64
 4   previous_points  67472 non-null  float64
 5   rank_change      67472 non-null  int64  
 6   confederation    67472 non-null  str    
 7   rank_date        67472 non-null  str    
dtypes: float64(3), int64(1), str(4)
memory usage: 4.1 MB


In [7]:
results_df["date"] = pd.to_datetime(results_df["date"])
rankings_df["rank_date"] = pd.to_datetime(rankings_df["rank_date"])

In [8]:
# there's a mismatch between names of our two csv files which might have caused the NaN in home nd away rank
name_fix = {
    'United States'       : 'USA',
    'Ivory Coast'         : 'Côte d\'Ivoire',
    'Republic of Ireland' : 'Ireland',
    'South Korea'         : 'Korea Republic',
    'North Korea'         : 'Korea DPR',
    'Czech Republic'      : 'Czechia',
    'Macedonia'           : 'North Macedonia',
    'Swaziland'           : 'Eswatini',
    'Cape Verde'          : 'Cape Verde Islands',
}

results_df['home_team'] = results_df['home_team'].replace(name_fix)
results_df['away_team'] = results_df['away_team'].replace(name_fix)

In [9]:
results_df["tournament"].value_counts()

tournament
Friendly                                18388
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1036
                                        ...  
Copa Confraternidad                         1
Benedikt Fontana Cup                        1
ConIFA Challenger Cup                       1
CONIFA World Cup qualification              1
South Asian Super Cup                       1
Name: count, Length: 200, dtype: int64

In [10]:
competitive = ["FIFA World Cup qualification", "UEFA Euro qualification", "African Cup of Nations qualification",
               "FIFA World Cup", "Copa América", "African Cup of Nations", "AFC Asian Cup qualification",
               "UEFA Nations League" , "CECAFA Cup"]

results_df = results_df[results_df['tournament'].isin(competitive)]

In [11]:
results_df.shape

(18779, 9)

In [12]:
#create target variable
def get_results(row):
    if row["home_score"] > row["away_score"]:
        return 2 #home win
    elif row["home_score"] == row["away_score"]:
        return 1 #draw
    else:
        return 0 #away win

results_df["result"] = results_df.apply(get_results, axis=1) #create a new results row applying the match win result

In [13]:
#merge home team rankings to results dataframe in a new dataframe called df_home
df_home = pd.merge_asof(
    results_df.sort_values('date'),
    rankings_df[['rank_date','country_full','rank','total_points']].rename(columns={
        'rank_date'    : 'date',
        'country_full' : 'home_team',
        'rank'         : 'home_rank',
        'total_points' : 'home_points'
    }).sort_values('date'),
    on='date',
    by='home_team'
)


In [14]:
#merge away team rankings
df_final = pd.merge_asof(
    df_home.sort_values('date'),
    rankings_df[['rank_date','country_full','rank','total_points']].rename(columns={
        'rank_date'    : 'date',
        'country_full' : 'away_team',
        'rank'         : 'away_rank',
        'total_points' : 'home_points'
    }).sort_values('date'),
    on='date',
    by='away_team'
)

In [15]:
df_final.shape

(18779, 14)

In [16]:
df_final.head(10) #first 10 rows

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,home_rank,home_points_x,away_rank,home_points_y
0,1916-07-02,Chile,Uruguay,0.0,4.0,Copa América,Buenos Aires,Argentina,True,0,NaN,NaN,NaN,NaN
1,1916-07-06,Argentina,Chile,6.0,1.0,Copa América,Buenos Aires,Argentina,False,2,NaN,NaN,NaN,NaN
2,1916-07-08,Brazil,Chile,1.0,1.0,Copa América,Buenos Aires,Argentina,True,1,NaN,NaN,NaN,NaN
3,1916-07-10,Argentina,Brazil,1.0,1.0,Copa América,Buenos Aires,Argentina,False,1,NaN,NaN,NaN,NaN
4,1916-07-12,Brazil,Uruguay,1.0,2.0,Copa América,Buenos Aires,Argentina,True,0,NaN,NaN,NaN,NaN
5,1916-07-17,Argentina,Uruguay,0.0,0.0,Copa América,Avellaneda,Argentina,False,1,NaN,NaN,NaN,NaN
6,1917-09-30,Uruguay,Chile,4.0,0.0,Copa América,Montevideo,Uruguay,False,2,NaN,NaN,NaN,NaN
7,1917-10-03,Argentina,Brazil,4.0,2.0,Copa América,Montevideo,Uruguay,True,2,NaN,NaN,NaN,NaN
8,1917-10-06,Argentina,Chile,1.0,0.0,Copa América,Montevideo,Uruguay,True,2,NaN,NaN,NaN,NaN
9,1917-10-07,Uruguay,Brazil,4.0,0.0,Copa América,Montevideo,Uruguay,False,2,NaN,NaN,NaN,NaN


In [17]:
df_final[["home_rank", "away_rank"]].isna().sum()

home_rank    5774
away_rank    5863
dtype: int64

In [18]:
post_1992 = df_final[df_final["date"] > '1992-01-01']
print(post_1992[['date','home_team','away_team','home_rank','away_rank']].head())

           date home_team      away_team  home_rank  away_rank
4797 1992-01-12  Cameroon        Morocco        NaN        NaN
4798 1992-01-12   Senegal        Nigeria        NaN        NaN
4799 1992-01-13   Algeria  Côte d'Ivoire        NaN        NaN
4800 1992-01-13     Egypt         Zambia        NaN        NaN
4801 1992-01-14   Morocco       DR Congo        NaN        NaN


In [19]:
# Check what team names look like in each CSV
print("results.csv sample teams:")
print(results_df['home_team'].unique()[:20])



results.csv sample teams:
<StringArray>
[      'Chile',   'Argentina',      'Brazil',     'Uruguay',    'Paraguay',
     'Bolivia',        'Peru',     'Belgium',      'France',      'Sweden',
   'Lithuania',  'Yugoslavia',      'Poland', 'Switzerland',       'Haiti',
     'Ireland',      'Mexico',  'Luxembourg',       'Spain',       'Egypt']
Length: 20, dtype: str


In [20]:
df_final.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,home_rank,home_points_x,away_rank,home_points_y
0,1916-07-02,Chile,Uruguay,0.0,4.0,Copa América,Buenos Aires,Argentina,True,0,NaN,NaN,NaN,NaN
1,1916-07-06,Argentina,Chile,6.0,1.0,Copa América,Buenos Aires,Argentina,False,2,NaN,NaN,NaN,NaN
2,1916-07-08,Brazil,Chile,1.0,1.0,Copa América,Buenos Aires,Argentina,True,1,NaN,NaN,NaN,NaN
3,1916-07-10,Argentina,Brazil,1.0,1.0,Copa América,Buenos Aires,Argentina,False,1,NaN,NaN,NaN,NaN
4,1916-07-12,Brazil,Uruguay,1.0,2.0,Copa América,Buenos Aires,Argentina,True,0,NaN,NaN,NaN,NaN


In [21]:
post_1993 = df_final[df_final['date'] > '1993-01-01'] #fifa rankings didnt exist before 1992
print(post_1993[['date','home_team','away_team','home_rank','away_rank']].head(10))

           date     home_team      away_team  home_rank  away_rank
5038 1993-01-10        Angola       Zimbabwe      102.0       54.0
5039 1993-01-10      DR Congo       Cameroon        NaN       22.0
5040 1993-01-10       Senegal        Algeria       51.0       30.0
5041 1993-01-16  South Africa        Nigeria      124.0       13.0
5042 1993-01-16      Tanzania         Zambia       80.0       32.0
5043 1993-01-17         Benin        Tunisia      127.0       38.0
5044 1993-01-17      Ethiopia        Morocco       85.0       41.0
5045 1993-01-17      Botswana  Côte d'Ivoire      139.0       27.0
5046 1993-01-17       Burundi        Algeria      100.0       30.0
5047 1993-01-17       Namibia     Madagascar      158.0       74.0


In [22]:
df_final["home_rank"].notna().sum()

np.int64(13005)

In [23]:
df_final["away_rank"].notna().sum()

np.int64(12916)

In [24]:
df_final['home_rank'].isna().sum() #pre-1992 NaN cols will be dropped

np.int64(5774)

Feature engineering

In [25]:
df_final["result"] = df_final.apply(get_results, axis=1) #create target variable for final dataframe


In [26]:
df_final["rank_diff"] = df_final["home_rank"] - df_final["away_rank"]
df_final["points_diff"] = df_final["home_points_x"] - df_final["home_points_y"]
df_final["is_neutral"] = df_final["neutral"].astype(int)

In [27]:
df_final[["rank_diff", "points_diff", "is_neutral"]].head(10)

,rank_diff,points_diff,is_neutral
0,NaN,NaN,1
1,NaN,NaN,0
2,NaN,NaN,1
3,NaN,NaN,0
4,NaN,NaN,1
5,NaN,NaN,0
6,NaN,NaN,0
7,NaN,NaN,1
8,NaN,NaN,1
9,NaN,NaN,0


In [28]:
df_final.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,home_rank,home_points_x,away_rank,home_points_y,rank_diff,points_diff,is_neutral
0,1916-07-02,Chile,Uruguay,0.0,4.0,Copa América,Buenos Aires,Argentina,True,0,NaN,NaN,NaN,NaN,NaN,NaN,1
1,1916-07-06,Argentina,Chile,6.0,1.0,Copa América,Buenos Aires,Argentina,False,2,NaN,NaN,NaN,NaN,NaN,NaN,0
2,1916-07-08,Brazil,Chile,1.0,1.0,Copa América,Buenos Aires,Argentina,True,1,NaN,NaN,NaN,NaN,NaN,NaN,1
3,1916-07-10,Argentina,Brazil,1.0,1.0,Copa América,Buenos Aires,Argentina,False,1,NaN,NaN,NaN,NaN,NaN,NaN,0
4,1916-07-12,Brazil,Uruguay,1.0,2.0,Copa América,Buenos Aires,Argentina,True,0,NaN,NaN,NaN,NaN,NaN,NaN,1


In [30]:
df_final = df_final.sort_values("date").reset_index(drop=True) #resets row nos to 0,1,2,3... after sorting

from collections import defaultdict
team_matches = defaultdict(list) #defaultdict(list) is like a normal Python dictionary except if you access a key that doesn't exist yet, it automatically creates an empty list for it.

for _,row in df_final.iterrows():
    team_matches[row["home_team"]].append({
        "date": row["date"],
        "goals_score": row["home_score"],
        "goals_conceded": row["away_score"],
        'win' : 1 if row["home_score"] > row["away_score"] else 0,
    })
    team_matches[row["away_team"]].append({
        "date": row["date"],
        "goals_score": row["away_score"],
        "goals_conceded": row["home_score"],
        'win' : 1 if row["home_score"] < row["away_score"] else 0,
    })

After this loop, team_matches looks like:

{
  "Brazil":    [match1, match2, match3, ...],

  "Argentina": [match1, match2, ...],

  ...
}

In [31]:
def get_current_form(team_name, cutoff_date):
    past_matches = [m for m in team_matches[team_name] if m["date"] > cutoff_date][-5:] #directly searches from the dictionary, -5: takes in the last 5 games
    if len(past_matches) == 0:
        return 0.5,0,0 #win_rate, goals_avg, conceded avg
    win_rate = sum(m["win"] for m in past_matches) / len(past_matches)
    avg_goals = sum(m["goals_score"] for m in past_matches) / len(past_matches)
    goals_conceded = sum(m["goals_conceded"] for m in past_matches) / len(past_matches)

    return win_rate, avg_goals, goals_conceded

home_winrate, home_avg_goals, home_goals_conceded = [], [], []
away_winrate, away_avg_goals, away_goals_conceded = [], [], []

for _,row in df_final.iterrows():
    hwr,hag,hcg = get_current_form(row["home_team"], row["date"])
    home_winrate.append(hwr)
    home_avg_goals.append(hag)
    home_goals_conceded.append(hcg)

    awr,aag,acg = get_current_form(row["away_team"], row["date"])
    away_winrate.append(awr)
    away_avg_goals.append(aag)
    away_goals_conceded.append(acg)

In [32]:
df_final["home_win_rate"] = home_winrate
df_final["home_avg_goals"] = home_avg_goals
df_final["home_goals_conceded"] = home_goals_conceded

df_final["away_win_rate"] = away_winrate
df_final["away_avg_goals"] = away_avg_goals
df_final["away_goals_conceded"] = away_goals_conceded

In [37]:
df_final.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,...,home_points_y,rank_diff,points_diff,is_neutral,home_win_rate,home_avg_goals,home_goals_conceded,away_win_rate,away_avg_goals,away_goals_conceded
0,1916-07-02,Chile,Uruguay,0.0,4.0,Copa América,Buenos Aires,Argentina,True,0,...,NaN,NaN,NaN,1,0.0,0.0,1.2,0.2,NaN,NaN
1,1916-07-06,Argentina,Chile,6.0,1.0,Copa América,Buenos Aires,Argentina,False,2,...,NaN,NaN,NaN,0,0.4,NaN,NaN,0.0,0.0,1.2
2,1916-07-08,Brazil,Chile,1.0,1.0,Copa América,Buenos Aires,Argentina,True,1,...,NaN,NaN,NaN,1,0.2,NaN,NaN,0.0,0.0,1.2
3,1916-07-10,Argentina,Brazil,1.0,1.0,Copa América,Buenos Aires,Argentina,False,1,...,NaN,NaN,NaN,0,0.4,NaN,NaN,0.2,NaN,NaN
4,1916-07-12,Brazil,Uruguay,1.0,2.0,Copa América,Buenos Aires,Argentina,True,0,...,NaN,NaN,NaN,1,0.2,NaN,NaN,0.2,NaN,NaN


In [38]:
features = [
    'home_rank', 'away_rank', 'rank_diff',
    'points_diff', 'is_neutral',
    'home_win_rate', 'home_avg_goals', 'home_goals_conceded',
    'away_win_rate', 'away_avg_goals', 'away_goals_conceded',
]


In [39]:
df_final = df_final.dropna(subset=features)

In [40]:
df_final[features].head()

,home_rank,away_rank,rank_diff,points_diff,is_neutral,home_win_rate,home_avg_goals,home_goals_conceded,away_win_rate,away_avg_goals,away_goals_conceded
5038,102.0,54.0,48.0,-17.0,0,0.0,0.8,1.0,0.0,0.8,1.4
5042,80.0,32.0,48.0,-23.0,0,0.0,0.6,1.2,0.2,0.4,1.0
5047,101.0,54.0,47.0,-17.0,0,0.2,0.2,1.0,0.0,0.8,1.4
5048,107.0,55.0,52.0,-18.0,0,0.4,1.0,1.8,0.2,1.4,2.2
5049,104.0,22.0,82.0,-33.0,0,0.0,1.0,2.6,0.6,1.2,1.0


In [41]:
df_final.shape

(5837, 23)

In [42]:
print('result' in df_final.columns)

True
